In [1]:
# Cell 1: imports library 
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Cell 2: paths setups & load dataset
pd.set_option("display.max_colwidth", 120)

df = pd.read_csv("../data/raw/bitext_raw.csv") 
print("Loaded successfully")
print(df.head())

Loaded successfully
   flags                                                   instruction  \
0      B              question about cancelling order {{Order Number}}   
1    BQZ    i have a question about cancelling oorder {{Order Number}}   
2   BLQZ               i need help cancelling puchase {{Order Number}}   
3     BL                    I need to cancel purchase {{Order Number}}   
4  BCELN  I cannot afford this order, cancel purchase {{Order Number}}   

  category        intent  \
0    ORDER  cancel_order   
1    ORDER  cancel_order   
2    ORDER  cancel_order   
3    ORDER  cancel_order   
4    ORDER  cancel_order   

                                                                                                                  response  
0  I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the...  
1  I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Pleas

In [3]:
# Cell 3: dimensions, columns, types
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes\n:", df.dtypes)

Shape: (26872, 5)

Columns: ['flags', 'instruction', 'category', 'intent', 'response']

Dtypes
: flags          str
instruction    str
category       str
intent         str
response       str
dtype: object


In [5]:
# Cell 4: memory, missing values, duplicates
print("Memroy usage (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 2))
print("\nMissing values per column:\n", df.isna().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate customer message:", df.iloc[:, df.columns.get_loc("instruction")].duplicated().sum()
      if "instruction" in df.columns else "no 'instruction' column")

Memroy usage (MB): 20.06

Missing values per column:
 flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Exact duplicate rows: 0
Duplicate customer message: 2237


In [6]:
# Cell 5: identify the target labels
for col in ["category", "intent"]:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts(), "\n")
    else:
        print(f"Column '{col}' not found")

category: 11 unique values
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64 

intent: 27 unique values
intent
check_invoice               1000
complaint                   1000
contact_customer_service    1000
edit_account                1000
switch_account              1000
check_payment_methods        999
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
set_up_shippi

In [7]:
# Cell 6: how categories and intents relate
mapping = df.groupby("category")["intent"].unique()
for cat, intents in mapping.items():
    print(f"{cat} -> {sorted(intents)}")

ACCOUNT -> ['create_account', 'delete_account', 'edit_account', 'recover_password', 'registration_problems', 'switch_account']
CANCEL -> ['check_cancellation_fee']
CONTACT -> ['contact_customer_service', 'contact_human_agent']
DELIVERY -> ['delivery_options', 'delivery_period']
FEEDBACK -> ['complaint', 'review']
INVOICE -> ['check_invoice', 'get_invoice']
ORDER -> ['cancel_order', 'change_order', 'place_order', 'track_order']
PAYMENT -> ['check_payment_methods', 'payment_issue']
REFUND -> ['check_refund_policy', 'get_refund', 'track_refund']
SHIPPING -> ['change_shipping_address', 'set_up_shipping_address']
SUBSCRIPTION -> ['newsletter_subscription']


In [8]:
# Cell 7: the flags column and a few samples
print(df["flags"].value_counts().head(15))
df.sample(5, random_state=42)[["instruction", "intent", "flags"]]

flags
BL       5212
BLQ      2467
BIL      2138
BLM      1297
BILQ     1057
BLQZ      970
BLZ       902
BKL       862
BLMQ      600
BEL       533
BILM      521
BCL       484
BCIL      427
BILQZ     412
BILP      363
Name: count, dtype: int64


,instruction,intent,flags
9329,I can't talk with a human agent,contact_human_agent,BLZ
4160,I have got to locate hte bills from {{Person Name}},check_invoice,BLMZ
18500,"I cannot pay, help me to inform of a problem with payments",payment_issue,BCELM
8840,I want help speaking to customer service,contact_customer_service,BL
5098,I try to see th accepted payment options,check_payment_methods,BLZ


In [10]:
# Cell 8: message length preview 
df["n_words"] = df["instruction"].str.split().str.len()
print(df["n_words"].describe())
df = df.drop(columns="n_words")

count    26872.000000
mean         8.690979
std          2.605004
min          1.000000
25%          7.000000
50%          9.000000
75%         11.000000
max         16.000000
Name: n_words, dtype: float64


In [11]:
# Cell 9: Are duplicated messages a problem?
dup_mask = df["instruction"].duplicated(keep=False)
dups = df[dup_mask]
print("Rows involved:", dup_mask.sum(), "| unique duplicated texts:", dups["instruction"].nunique())

conflict = dups.groupby("instruction")["intent"].nunique()
print("Texts that appear with MORE THAN ONE intent:", (conflict > 1).sum())

dups.sort_values("instruction").head(10)[["instruction", "intent", "flags"]]

Rows involved: 3226 | unique duplicated texts: 989
Texts that appear with MORE THAN ONE intent: 0


,instruction,intent,flags
25320,ETA of order {{Order Number}},track_order,BK
25447,ETA of order {{Order Number}},track_order,BK
25557,ETA of order {{Order Number}},track_order,BK
25605,ETA of order {{Order Number}},track_order,BK
25864,ETA of order {{Order Number}},track_order,BK
24880,ETA of purchase {{Order Number}},track_order,BKLQ
25846,ETA of purchase {{Order Number}},track_order,BKLQ
235,I am trying to cancel order {{Order Number}},cancel_order,B
636,I am trying to cancel order {{Order Number}},cancel_order,B
418,I am trying to cancel purchaes {{Order Number}},cancel_order,BLZ


In [12]:
# Cell 10: how many messages contain placeholders?
ph = df["instruction"].str.findall(r"\{\{.*?\}\}").explode().value_counts()
print(ph.head(15))
print("Share of message with a placeholder:", round(df["instruction"].str.contains(r"\{\{", regex=True).mean(), 3))

instruction
{{Order Number}}        2907
{{Account Type}}        1011
{{Person Name}}          887
{{Account Category}}     822
{{Refund Amount}}        624
{{Currency Symbol}}      372
{{Delivery City}}        234
{{Delivery Country}}     177
{{Invoice Number}}         8
Name: count, dtype: int64
Share of message with a placeholder: 0.248
